# LION Experiment Plots

Fetches the `final_results` group from W&B project `jfcevallos/LION`,
groups the 90 runs (9 agents × 10 seeds) by the `agent` config parameter,
and plots mean ± 95 % CI across seeds for key metrics.

**Requires** a `.env` file in the same directory containing:
```
WANDB_API_KEY=<your_key>
```

In [ ]:
import os
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from dotenv import load_dotenv

import wandb

warnings.filterwarnings('ignore', category=FutureWarning)

## Utils

In [ ]:
# ── Typography ──────────────────────────────────────────────────────────────
title_font  = {'weight': 'bold', 'size': 22}
axis_font   = {'weight': 'bold', 'size': 14}
legend_font = {'weight': 'bold', 'size': 11}


def get_color_map(agents):
    """
    Prefer the hard-coded per-agent colours used in synthetic_lion.py,
    falling back to tab10 for any unknown name.
    """
    hardcoded = {
        'DDQN':         '#2196F3',
        'Dueling-DDQN': '#00BCD4',
        'PPO':          '#FF5722',
        'SAC':          '#8BC34A',
        'DAI-P':        '#E91E63',
        'DAI-A':        '#FF9800',
        'DAI-SA':       '#9C27B0',
        'DAI-F':        '#4CAF50',
        'Break-Even':   '#607D8B',
    }
    fallback = itertools.cycle(plt.colormaps.get_cmap('tab10').colors)
    return {a: hardcoded.get(a, next(fallback)) for a in agents}


def compute_ci_band(histories: list[pd.Series], alpha: float = 0.95):
    """
    Given a list of per-seed time-series (indexed by episode step),
    return (mean, lower_ci, upper_ci) aligned to a common step range.

    Uses a *t*-distribution-based 95 % CI:
        mean ± t_{n-1, 0.975} * std / sqrt(n)
    Falls back to ±std when n < 2.
    """
    from scipy import stats as scipy_stats

    if not histories:
        return None, None, None

    # Align all series to a shared integer index (episode number)
    combined = pd.concat(histories, axis=1)
    combined = combined.apply(pd.to_numeric, errors='coerce')

    mean = combined.mean(axis=1)
    std  = combined.std(axis=1, ddof=1)
    n    = combined.notna().sum(axis=1)

    # t multiplier per row (gracefully handles n=1 by returning nan CI)
    t_mult = pd.Series(
        [scipy_stats.t.ppf((1 + alpha) / 2, df=max(ni - 1, 1)) if ni > 1 else float('nan')
         for ni in n],
        index=mean.index
    )
    margin = t_mult * std / np.sqrt(n)
    return mean, mean - margin, mean + margin


def _style_ax(ax, title, xlabel, ylabel, ylim=None):
    """Apply consistent styling to a subplot axis."""
    ax.set_title(title, **title_font)
    ax.set_xlabel(xlabel, **axis_font)
    ax.set_ylabel(ylabel, **axis_font)
    ax.tick_params(axis='both', labelsize=axis_font['size'])
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('bold')
    if ylim:
        ax.set_ylim(ylim)
    ax.grid(True, alpha=0.4)


def plot_metric_grid(
    agent_data: dict,
    metrics: list,
    n_cols: int = 2,
    title: str = '',
    smooth_window: int = 10,
    figsize_per_cell=(10, 4),
    agent_subset=None,
):
    """
    Draws a grid of subplots, one per metric in `metrics`.

    agent_data  : {agent_name: pd.DataFrame (index=episode, columns=metrics)}
    metrics     : list of dicts with keys: key, title, ylim (opt), xlabel (opt)
    n_cols      : number of grid columns
    smooth_window : rolling-mean window applied *before* CI aggregation is shown
    agent_subset  : if given, only plot these agents
    """
    agents_to_plot = agent_subset if agent_subset else list(agent_data.keys())
    color_map = get_color_map(agents_to_plot)

    n_metrics = len(metrics)
    n_rows    = (n_metrics + n_cols - 1) // n_cols
    fig_w, fig_h = figsize_per_cell
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(fig_w * n_cols, fig_h * n_rows),
        squeeze=False
    )
    axes_flat = axes.flatten()

    for i, metric in enumerate(metrics):
        ax  = axes_flat[i]
        key = metric['key']

        for agent in agents_to_plot:
            runs_df = agent_data.get(agent)
            if runs_df is None or key not in runs_df.columns:
                continue

            # Each seed is a column; smooth each seed independently
            smoothed = runs_df[key].rolling(smooth_window, min_periods=1).mean()

            # Rebuild as list of per-seed series for CI computation
            seed_series = [smoothed[col].dropna() for col in smoothed.columns]
            mean, lo, hi = compute_ci_band(seed_series)

            if mean is None:
                continue

            color = color_map[agent]
            ax.plot(mean.index, mean.values, label=agent, color=color,
                    linewidth=2, alpha=0.9)
            if lo is not None:
                ax.fill_between(lo.index, lo.values, hi.values,
                                color=color, alpha=0.18)

        _style_ax(
            ax,
            title=metric.get('title', key),
            xlabel=metric.get('xlabel', 'Episode'),
            ylabel=metric.get('ylabel', metric.get('title', key)),
            ylim=metric.get('ylim'),
        )

    # Hide unused subplot cells
    for j in range(n_metrics, len(axes_flat)):
        axes_flat[j].set_visible(False)

    # Shared legend below the figure
    handles = [Line2D([0], [0], color=color_map[a], lw=2, label=a)
               for a in agents_to_plot if a in color_map]
    fig.legend(
        handles=handles,
        prop=legend_font,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.04),
        ncol=min(len(agents_to_plot), 5),
        frameon=True,
    )

    if title:
        fig.suptitle(title, weight='bold', size=26, y=1.01)

    plt.tight_layout()
    plt.show()

## Setup: W&B API

In [ ]:
load_dotenv()   # reads WANDB_API_KEY from .env

api = wandb.Api()
print('W&B API initialised')

## Fetch Runs  (group = `final_results`)

In [ ]:
WANDB_PROJECT = 'jfcevallos/LION'
GROUP         = 'final_results'
N_SAMPLES     = 10_000   # history samples per run (W&B cap; raise if needed)

all_runs = api.runs(
    WANDB_PROJECT,
    filters={'group': GROUP},
)
print(f'Found {len(all_runs)} runs in group "{GROUP}"')

In [ ]:
# ── Fetch histories, keyed by (agent, seed) ─────────────────────────────────
#
# W&B config stores 'agent' and 'seed' as custom params set in synthetic_lion.py:
#   config={..., 'seed': seed, 'agent': agent_name}

raw_runs = {}   # {(agent_name, seed): pd.DataFrame}

for run in all_runs:
    agent_name = run.config.get('agent', run.job_type or 'unknown')
    seed       = run.config.get('seed', -1)
    print(f'  fetching  {agent_name:12s}  seed={seed}  ({run.id})')

    hist = run.history(samples=N_SAMPLES)
    if '_step' in hist.columns:
        hist = hist.set_index('_step')
    elif hist.index.name != '_step':
        hist.index.name = 'episode'

    raw_runs[(agent_name, seed)] = hist

print(f'\nFetched {len(raw_runs)} runs total')

In [ ]:
# ── Organise into agent_data:
#    {agent_name: {metric_col: DataFrame(index=episode, columns=seeds)}}
#
# We keep a *wide* DataFrame per agent where each column is one seed.
# This makes vectorised CI computation straightforward.

from collections import defaultdict

# First pass: discover all agents and all metric columns
agents_found = sorted({k[0] for k in raw_runs})
all_cols     = sorted({col for df in raw_runs.values() for col in df.columns})

print('Agents found:', agents_found)
print('\nMetric columns available:')
for c in all_cols:
    print(' ', c)

# Second pass: build per-agent wide DataFrames  {agent: {col: wide_df}}
agent_data = {}   # agent_name -> wide DataFrame (index=episode, columns=seed_id)

for agent in agents_found:
    seed_dfs = {seed: df for (ag, seed), df in raw_runs.items() if ag == agent}
    if not seed_dfs:
        continue

    # Build one wide df per metric column
    # We store a single wide df where column tuples are (metric, seed)
    all_seed_dfs = [df.add_suffix(f'__seed{s}') for s, df in seed_dfs.items()]
    wide = pd.concat(all_seed_dfs, axis=1).sort_index()
    agent_data[agent] = wide

print(f'\nBuilt wide DataFrames for {len(agent_data)} agents')

In [ ]:
# ── Helper: extract per-metric wide DataFrame for a specific metric key ───
#
# Returns {agent: DataFrame(index=episode, columns=seed_ids)}

def get_metric_wide(metric_key: str) -> dict:
    result = {}
    for agent, wide in agent_data.items():
        cols = [c for c in wide.columns if c.startswith(metric_key + '__seed')]
        if not cols:
            continue
        result[agent] = wide[cols].copy()
        result[agent].columns = [c.split('__seed')[1] for c in cols]  # seed id as col name
    return result


# Quick sanity check
rew_wide = get_metric_wide('reward')
print('reward wide shapes:', {a: df.shape for a, df in rew_wide.items()})

## Plotting helper (CI-aware)

Rather than rolling quantiles within a single run (as in the Tiger notebook),  
we compute a proper **95 % confidence interval across the 10 seeds** for each agent.

In [ ]:
from scipy import stats as scipy_stats

def plot_metric_grid_v2(
    metrics: list,
    n_cols: int = 2,
    suptitle: str = '',
    smooth_window: int = 15,
    figsize_per_cell=(10, 4),
    agent_subset=None,
    ci_alpha: float = 0.95,
):
    """
    metrics: list of dicts
        - key      : wandb metric name
        - title    : subplot title
        - ylim     : (lo, hi) or None
        - xlabel   : x-axis label (default 'Episode')
        - ylabel   : y-axis label (default title)
    """
    agents_to_plot = agent_subset if agent_subset else agents_found
    color_map      = get_color_map(agents_to_plot)

    n_metrics = len(metrics)
    n_rows    = (n_metrics + n_cols - 1) // n_cols
    fw, fh    = figsize_per_cell
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(fw * n_cols, fh * n_rows),
                             squeeze=False)
    axes_flat = axes.flatten()

    for i, metric in enumerate(metrics):
        ax  = axes_flat[i]
        key = metric['key']

        metric_wide = get_metric_wide(key)

        for agent in agents_to_plot:
            wide = metric_wide.get(agent)
            if wide is None or wide.empty:
                continue

            # Convert to numeric and smooth each seed
            wide_num = wide.apply(pd.to_numeric, errors='coerce')
            wide_smoothed = wide_num.rolling(smooth_window, min_periods=1).mean()

            # CI across seeds
            vals  = wide_smoothed.values      # shape (episodes, n_seeds)
            idx   = wide_smoothed.index
            n     = np.sum(~np.isnan(vals), axis=1)
            mean  = np.nanmean(vals, axis=1)
            std   = np.nanstd(vals, axis=1, ddof=1)

            # t-based margin
            t_mult = np.where(
                n > 1,
                scipy_stats.t.ppf((1 + ci_alpha) / 2, df=np.maximum(n - 1, 1)),
                np.nan
            )
            margin = t_mult * std / np.sqrt(np.maximum(n, 1))

            color = color_map[agent]
            ax.plot(idx, mean, label=agent, color=color, linewidth=2, alpha=0.9)
            ax.fill_between(idx, mean - margin, mean + margin,
                            color=color, alpha=0.18)

        _style_ax(
            ax,
            title=metric.get('title', key),
            xlabel=metric.get('xlabel', 'Episode'),
            ylabel=metric.get('ylabel', metric.get('title', key)),
            ylim=metric.get('ylim'),
        )

    for j in range(n_metrics, len(axes_flat)):
        axes_flat[j].set_visible(False)

    # Single legend below the grid
    handles = [Line2D([0], [0], color=color_map[a], lw=2, label=a)
               for a in agents_to_plot if a in color_map]
    # Add CI shading swatch to legend
    ci_patch = mpatches.Patch(alpha=0.3, color='grey',
                               label=f'{int(ci_alpha*100)}% CI (across seeds)')
    fig.legend(
        handles=handles + [ci_patch],
        prop=legend_font,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.06),
        ncol=min(len(agents_to_plot) + 1, 5),
        frameon=True,
    )

    if suptitle:
        fig.suptitle(suptitle, weight='bold', size=26, y=1.02)

    plt.tight_layout()
    plt.show()

---
## Grid Plot 1 — Core Performance Metrics

Reward, win rate, number of labels bought, label-A purchase rate, final budget, and training loss  
averaged across 10 seeds with 95 % confidence intervals.

In [ ]:
perf_metrics = [
    {'key': 'reward',       'title': 'Mean Step Reward',         'ylim': None},
    {'key': 'win',          'title': 'Win Rate',                 'ylim': (-0.05, 1.05), 'ylabel': 'Fraction'},
    {'key': 'n_labels',     'title': 'Labels Bought / Episode',  'ylim': (0, None)},
    {'key': 'label_A',      'title': 'Label-A Purchase Rate',    'ylim': (-0.05, 1.05), 'ylabel': 'Fraction'},
    {'key': 'budget_final', 'title': 'Final Episode Budget',     'ylim': None},
    {'key': 'loss',         'title': 'Agent Training Loss',      'ylim': (0, None)},
]

plot_metric_grid_v2(
    metrics=perf_metrics,
    n_cols=2,
    suptitle='LION — Core Performance Metrics  (mean ± 95 % CI across seeds)',
    smooth_window=20,
)

---
## Grid Plot 2 — Unknown-Cluster Confidence Tracking

The inference module maintains an EMA confidence estimate for each unknown cluster  
(A, B, C, D).  Buying a label resets the confidence to zero; it then rises again  
as new unlabelled traffic arrives.

In [ ]:
conf_metrics = [
    {'key': 'unk_confs/A', 'title': 'Unknown-Cluster A Confidence', 'ylim': (0, 1)},
    {'key': 'unk_confs/B', 'title': 'Unknown-Cluster B Confidence', 'ylim': (0, 1)},
    {'key': 'unk_confs/C', 'title': 'Unknown-Cluster C Confidence', 'ylim': (0, 1)},
    {'key': 'unk_confs/D', 'title': 'Unknown-Cluster D Confidence', 'ylim': (0, 1)},
]

plot_metric_grid_v2(
    metrics=conf_metrics,
    n_cols=2,
    suptitle='LION — Inference Module Unknown-Cluster Confidences  (mean ± 95 % CI)',
    smooth_window=15,
)

---
## Grid Plot 3 — Epistemic Gains  (DAI agents only)

DAI-based agents (DAI-P, DAI-A, DAI-SA, DAI-F) compute an epistemic gain per action.  
Higher gain → the action is more informative according to the agent's world model.

* **mean** — average epistemic gain across all three action types  
* **block / accept / buy_label** — per-action-type breakdown  
* **buy_A … buy_D** — epistemic value attributed to buying each specific label

In [ ]:
dai_agents = [a for a in agents_found if a.startswith('DAI')]
print('DAI agents:', dai_agents)

epistemic_metrics = [
    {'key': 'epistemic_gains/mean',      'title': 'Mean Epistemic Gain',         'ylim': (0, None)},
    {'key': 'epistemic_gains/buy_label', 'title': 'Epistemic Gain: buy_label',   'ylim': (0, None)},
    {'key': 'epistemic_gains/block',     'title': 'Epistemic Gain: block',       'ylim': (0, None)},
    {'key': 'epistemic_gains/accept',    'title': 'Epistemic Gain: accept',      'ylim': (0, None)},
    {'key': 'epistemic_gains/buy_A',     'title': 'Epistemic Gain: buy Label A', 'ylim': (0, None)},
    {'key': 'epistemic_gains/buy_B',     'title': 'Epistemic Gain: buy Label B', 'ylim': (0, None)},
]

plot_metric_grid_v2(
    metrics=epistemic_metrics,
    n_cols=2,
    suptitle='LION — Epistemic Gains (DAI agents, mean ± 95 % CI)',
    smooth_window=15,
    agent_subset=dai_agents,
)

---
## Grid Plot 4 (bonus) — Transition-Loss & Episode Length  (DAI agents with trans net)

DAI-P, DAI-A and DAI-F maintain a transition network; its loss (`trans_loss`) shows
how quickly the world model converges.  `n_steps` shows episode length over time.

In [ ]:
trans_agents = [a for a in agents_found if a in {'DAI-P', 'DAI-A', 'DAI-F'}]

trans_metrics = [
    {'key': 'trans_loss', 'title': 'Transition Network Loss', 'ylim': (0, None)},
    {'key': 'n_steps',    'title': 'Episode Length (steps)',  'ylim': (0, None)},
]

if trans_agents:
    plot_metric_grid_v2(
        metrics=trans_metrics,
        n_cols=2,
        suptitle='LION — Transition Net Loss & Episode Length  (mean ± 95 % CI)',
        smooth_window=15,
        agent_subset=trans_agents,
    )
else:
    print('No DAI agents with transition network found in this dataset.')